# LeapQuant KDA 教程 (2/5): 门从哪来 —— A_log / dt_bias 与两种展开式

KDA 的 per-K 衰减 $g$ 不是直接学出来的, 模型吐的是**原始激活** $a$, 再由两个小参数
展开成 $g$ (类似 Mamba 的 $\Delta t$):

- `A_log` $[H]$: 每头一个, 控制整体衰减速率;
- `dt_bias` $[H, K]$: **每头每键维度**一个偏置 —— 这正是"每列记忆时长"的来源。

本册验证 `kda_expand_gate` (M3.4a 的 host 展开实现) 与 AscendC 核内展开**逐位一致**。

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发仓 (kernel / 量化器所在)
torch.npu.set_device(0)
DEV = torch.device("npu:0")
torch.manual_seed(7)
print("device:", torch.npu.get_device_name(0))


from vllm_ascend.ops.triton.leapquant import kda_expand_gate

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


device: Ascend910_9382


/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:13: FutureWarning: triton.language.extra.cann.extension.atan2 is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.atan2 instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.atan2")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:35: FutureWarning: triton.language.extra.cann.extension.isfinited is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.isfinited instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.isfinited")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:49: FutureWarning: triton.language.extra.cann.extension.finitef is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.finitef instead.
  @_deprec

INFO 11-11 16:17:32 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-11 16:17:32 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-11 16:17:32 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-11 16:17:32 [__init__.py:272] Platform plugin ascend is activated


INFO 11-11 16:17:32 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


/usr/local/python3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 参数几何

In [2]:
H, K = 4, 16
a = torch.randn(2, 3, H, K, device=DEV) * 0.5          # 原始激活 [B,T,H,K] (模型投影输出)
a_log = torch.log(torch.rand(H, device=DEV) + 0.5)     # [H]
dt_bias = 0.1 * torch.randn(H, K, device=DEV)          # [H,K] -- 每列一个时间偏置
print("a:", tuple(a.shape), " A_log:", tuple(a_log.shape), " dt_bias:", tuple(dt_bias.shape))

[W1111 16:17:39.918089466 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


a: (2, 3, 4, 16)  A_log: (4,)  dt_bias: (4, 16)


## 2. softplus 变体 (Kimi-Linear 默认)

$$g = -e^{A_{log}} \cdot \mathrm{softplus}(a + \mathrm{dt\_bias})$$

负号保证 $g \le 0$ (只衰减不放大); `softplus` 在大输入时直接用 $x$ (threshold=20) 防止
$e^{x}$ 上溢 —— 和 flash-linear-attention 的实现一致。

In [3]:
g = kda_expand_gate(a, a_log, dt_bias)                 # [B,T,H,K]

x = a + dt_bias.view(1, 1, H, K)
softplus = torch.where(x <= 20.0, torch.log1p(torch.exp(x)), x)
manual = -torch.exp(a_log).view(1, 1, H, 1) * softplus

print("kda_expand_gate vs 手算 softplus: max|diff| =", (g - manual).abs().max().item())
print("g 范围: [%.3f, %.3f]  (全部 <= 0 ✓)" % (g.min(), g.max()))

kda_expand_gate vs 手算 softplus: max|diff| = 0.0
g 范围: [-2.385, -0.230]  (全部 <= 0 ✓)


## 3. 安全变体 (lower_bound)

softplus 变体的 $g$ 没有下界 —— 某些维度可能衰得**过狠** ( $e^{g} \to 0$, 记忆整列清零)。
安全变体换成 sigmoid, 把每列衰减率夹在 $[e^{lb}, 1)$:

$$g = \mathrm{lower\_bound} \cdot \sigma\left(e^{A_{log}} (a + \mathrm{dt\_bias})\right), \quad \mathrm{lb} \in [-5, 0)$$

In [4]:
lb = -3.0
g_safe = kda_expand_gate(a, a_log, dt_bias, lower_bound=lb)

manual_safe = lb * torch.sigmoid(torch.exp(a_log).view(1, 1, H, 1) * (a + dt_bias.view(1, 1, H, K)))
print("safe 变体 vs 手算: max|diff| =", (g_safe - manual_safe).abs().max().item())
print(f"g_safe ∈ [{g_safe.min():.3f}, {g_safe.max():.3f}], 下界 {lb} ✓")

def gate_at(av, lower_bound=None):
    return kda_expand_gate(torch.tensor([av], device=DEV).view(1, 1, 1, 1),
                           a_log[:1], dt_bias[:1, :1], lower_bound).item()

print("\n a 取值      softplus 变体        safe 变体 (lb=-3)")
for av in [-6.0, -2.0, 0.0, 2.0, 6.0]:
    print(f"  {av:5.1f}   {gate_at(av):9.4f}          {gate_at(av, lb):9.4f}")

safe 变体 vs 手算: max|diff| = 0.0
g_safe ∈ [-2.674, -0.368], 下界 -3.0 ✓

 a 取值      softplus 变体        safe 变体 (lb=-3)
   -6.0     -0.0033            -0.0006
   -2.0     -0.1708            -0.1632
    0.0     -0.9411            -1.4568
    2.0     -2.9246            -2.8180
    6.0     -8.3390            -2.9993


## 4. l2norm: q/k 进账本前先归一

KDA 的冲销公式假设 $\|k\| = 1$ (否则 $I - \beta kk^{\top}$ 不再是压缩), 所以 q/k 都要过
l2norm —— eps 加在**平方和**上, 与 fla 的 `l2norm_fwd` 一致。

In [5]:
from vllm.third_party.flash_linear_attention.ops.l2norm import l2norm_fwd

x = torch.randn(5, 8, device=DEV).to(torch.bfloat16)
manual = x / torch.sqrt((x * x).sum(-1, keepdim=True) + 1e-6)   # eps 在平方和里
print("l2norm_fwd vs 手算: max|diff| =", (l2norm_fwd(x.contiguous()).float() - manual.float()).abs().max().item())

l2norm_fwd vs 手算: max|diff| = 0.00390625


## 5. 终极验证: host 展开 == AscendC 核内展开

生产路径的 `fla_npu` AscendC `recurrent_kda` 把 gate 展开 + l2norm 都做在**核内**。
M3.4a 替换必须在 host 上复现同样的预处理。跑一步真实 binary 对比
(注意: AscendC 在旁流执行, 输入要 hold 住 + 同步, 见教程 5):

In [6]:
from vllm_ascend.ops.kda import run_recurrent_kda

H, DK, DV = 4, 128, 128
torch.manual_seed(52)
# 生产形态: [1, T=1, H, D] 现物化 (rearrange("n (h d) -> 1 n h d") 的同款),
# fp32 临时即刻消亡 —— 这个 op 对输入落址敏感, 详见教程 5
q = torch.randn(1, 1, H, DK, device=DEV).to(torch.bfloat16)
k = torch.randn(1, 1, H, DK, device=DEV).to(torch.bfloat16)
v = torch.randn(1, 1, H, DV, device=DEV).to(torch.bfloat16)
a = (torch.randn(1, 1, H, DK, device=DEV) * 0.5).to(torch.bfloat16)
a_log = torch.log(torch.rand(H, device=DEV) + 0.5)
dt_bias = 0.1 * torch.randn(H, DK, device=DEV)
beta = (torch.rand(1, 1, H, device=DEV) * 0.5 + 0.5).sigmoid()  # 已过 sigmoid (生产约定)

state = torch.zeros(4, H, DV, DK, device=DEV, dtype=torch.float32)  # 4 槽池, 用槽 1
indices = torch.tensor([1], device=DEV, dtype=torch.int32)
cu = torch.tensor([0, 1], device=DEV, dtype=torch.int32)

held = [q, k, v, a, beta]                                    # 防异步分配器复用 (教程 5 详述)
out_ac = run_recurrent_kda(q, k, v, a, beta, state, cu, indices, a_log, dt_bias, lower_bound=None)
torch.npu.synchronize()

# ---- host 复现: l2norm + gate 展开 + fp32 手写递推 --------------------
q_n = l2norm_fwd(q.contiguous()).float()                     # [1,1,H,DK]
k_n = l2norm_fwd(k.contiguous()).float()
g = kda_expand_gate(a.float(), a_log, dt_bias, None).float() # [1,1,H,DK]

S = torch.zeros(H, DV, DK, device=DEV)
S = S * torch.exp(g)[0, 0][:, None, :]                       # 逐列衰减 (从零状态出发, 仍是逐列)
sk = torch.einsum("hvk,hk->hv", S, k_n[0, 0])
vt = (v.float()[0, 0] - sk) * beta[0, 0][:, None]
S = S + torch.einsum("hv,hk->hvk", vt, k_n[0, 0])
o_host = torch.einsum("hvk,hk->hv", S, q_n[0, 0]) * (DK ** -0.5)

rel = ((out_ac[0, 0].float() - o_host).norm() / o_host.norm()).item()
print(f"AscendC 一步 vs host 复现: rel = {rel:.4f}  (bf16 输入噪声量级 ✓)")

AscendC 一步 vs host 复现: rel = 0.0017  (bf16 输入噪声量级 ✓)


## 小结 (KDA 教程 2)

- $g$ 由 `A_log` $[H]$ × `dt_bias` $[H,K]$ 从原始激活展开: dt_bias 的每列偏置就是"每列记忆时长"参数;
- softplus 变体 (Kimi-Linear 默认, 无下界) 与 `lower_bound` sigmoid 变体 (衰减率夹在 $[e^{lb},1)$) 都在
  `kda_expand_gate` 里, 与手算 fp32 逐位一致 (diff ≈ 0);
- q/k 进递推前要 l2norm (eps 在平方和里), β 已在 host 过 sigmoid;
- host 展开 + l2norm + fp32 递推与 AscendC 核内版本在 bf16 噪声内对齐 → 教程 5 的调用点替换成立。